# Make a list of everyone who emails you

This notebook reads the email export that **Google Takeout** saved in your Google Drive, and writes a file called `senders.csv` back into your Drive. It has one row per person who has sent you email: first name, last name, email address, and the original "From" text (so you can check the names were split correctly).

It runs on Google's computers, so nothing is downloaded to your laptop and your email never leaves your Google account.

**How to use it:** click the round play button (&#9654;) at the left of each grey box below, one at a time, from top to bottom. Wait for each one to finish (a green check mark appears) before clicking the next.

## Step 1 - Connect to your Google Drive

A pop-up will ask permission for this notebook to see your Drive. Click "Connect to Google Drive", choose your work account, and click "Continue" or "Allow".

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

## Step 2 - Find the Takeout export

Takeout saves its files in a folder called "Takeout" in your Drive. This lists the `.zip` files in it. You should see at least one file, ending in `.zip`, with a size in gigabytes.

In [ ]:
import glob
import os

export_files = sorted(glob.glob('/content/drive/MyDrive/Takeout*/*.zip'))

if not export_files:
    print("I could not find any .zip files in a Takeout folder in your Drive.")
    print("Check that the Takeout export has finished and was saved to Google Drive.")
else:
    for path in export_files:
        print(os.path.basename(path), ' ', round(os.path.getsize(path) / 1e9, 2), 'GB')

## Step 3 - Get the program that reads the email

This downloads a small program (`mbox_senders.py`) that reads the "From" line of every message in the export.

In [ ]:
import urllib.request

SCRIPT_URL = 'https://raw.githubusercontent.com/MorrisPearl/morris-lisp/main/gmail_from_header_export/mbox_senders.py'
urllib.request.urlretrieve(SCRIPT_URL, 'mbox_senders.py')
print('Got mbox_senders.py')

## Step 4 - Make the list

This can take 10 to 30 minutes or more for a big mailbox. Progress lines appear as it goes. **Keep this browser tab open and your laptop awake until it says "Wrote ... unique senders".**

Messages in Spam or Trash, and senders like `info@` and `no-reply@`, are left out.

In [ ]:
import shlex

zip_arguments = ' '.join(shlex.quote(path) for path in export_files)

!python3 mbox_senders.py {zip_arguments} -o /content/drive/MyDrive/senders.csv

## Step 5 - Take a look

This shows the first 20 rows, newest senders first.

In [ ]:
import pandas as pd

senders = pd.read_csv('/content/drive/MyDrive/senders.csv', keep_default_na=False)
print(len(senders), 'unique senders')
senders.head(20)

## Done

The list is in your Google Drive, in "My Drive", as `senders.csv`. To turn it into a spreadsheet: go to [drive.google.com](https://drive.google.com), right-click `senders.csv`, choose **Open with**, then **Google Sheets**.

**Please clean up afterwards:** the Takeout folder in your Drive holds a complete copy of all your email. Once you have the list, delete the "Takeout" folder, and then empty your Drive trash (the "Trash" link on the left side of Drive).